# 03. OIDC & Token Validation

## Background

OpenID Connect (OIDC) is an identity layer on top of OAuth 2.0. Where OAuth answers 'can this client access this resource?', OIDC answers 'who is this user?' It adds the **ID token** — a signed JWT containing identity claims — to the OAuth token response. OIDC powers Sign in with Google, GitHub, Apple, and every enterprise SSO.

Token validation is where many implementations fail. Skipping the signature check, the expiry check, or trusting the `alg` header blindly are common mistakes with catastrophic consequences.

## What You'll Learn

- ID token anatomy: header, payload, signature
- JWKS: how signing keys are published and rotated
- Mandatory validation steps: signature, issuer, audience, expiry, nonce
- Algorithm confusion (alg:none) defense
- Claim-based authorization

## Why This Matters

Every service accepting third-party identity tokens must validate them correctly. Missing one step is enough. The 2021 AWS Cognito misconfiguration allowed arbitrary audience bypass. Building the validator from scratch forces understanding each check and why it matters.

In [ ]:
import json, time, base64, hashlib, hmac, secrets
from dataclasses import dataclass, field
from typing import Optional

def b64url_encode(data):
    return base64.urlsafe_b64encode(data).rstrip(b'=').decode()

def b64url_decode(s):
    pad = 4 - len(s) % 4
    if pad != 4: s += '=' * pad
    return base64.urlsafe_b64decode(s)

def decode_jwt_parts(token):
    parts = token.split('.')
    if len(parts) != 3: raise ValueError(f'Invalid JWT: {len(parts)} parts')
    header  = json.loads(b64url_decode(parts[0]))
    payload = json.loads(b64url_decode(parts[1]))
    return header, payload, parts[2]


## 1. Minting a Signed ID Token (HMAC-SHA256 for demo)

In [ ]:
KEY_STORE = {
    'key-2024-01': b'secret_key_one_32bytes__________',
    'key-2024-02': b'secret_key_two_32bytes__________',
}
JWKS_KEYS = list(KEY_STORE.keys())

def sign_jwt(header, payload, key):
    h = b64url_encode(json.dumps(header, separators=(',',':')).encode())
    p = b64url_encode(json.dumps(payload, separators=(',',':')).encode())
    signing_input = f'{h}.{p}'.encode()
    sig = b64url_encode(hmac.new(key, signing_input, hashlib.sha256).digest())
    return f'{h}.{p}.{sig}'

def mint_id_token(user_id, email, roles, client_id, nonce, kid='key-2024-01'):
    now = int(time.time())
    h = {'alg': 'HS256', 'typ': 'JWT', 'kid': kid}
    p = {'iss': 'https://accounts.example.com', 'sub': user_id,
         'aud': client_id, 'iat': now, 'exp': now + 3600,
         'nonce': nonce, 'email': email, 'roles': roles, 'amr': ['mfa']}
    return sign_jwt(h, p, KEY_STORE[kid])

token = mint_id_token('usr_001', 'alice@example.com', ['analyst'], 'myapp', 'nonce_xyz')
print(f'Signed token (first 60): {token[:60]}...')
h, p, s = decode_jwt_parts(token)
print(f'Header:  {h}')
print(f'Payload: sub={p["sub"]} email={p["email"]} roles={p["roles"]}')


## 2. Full Validation Pipeline

Each check blocks a specific attack class:

| Step | Blocks |
|------|--------|
| Algorithm whitelist | alg:none, RS->HS confusion |
| Key lookup by kid | Key confusion, replay with old key |
| Signature verify | Payload tampering |
| Issuer check | Cross-issuer token reuse |
| Audience check | Token intended for different service |
| Expiry check | Replay of old tokens |
| Nonce check | Authorization code replay |

In [ ]:
@dataclass
class ValidationResult:
    valid:  bool
    claims: dict = field(default_factory=dict)
    errors: list = field(default_factory=list)

class OIDCTokenValidator:
    def __init__(self, issuer, audience, key_store):
        self.issuer   = issuer
        self.audience = audience
        self._keys    = key_store

    def validate(self, token, nonce=None):
        errors = []
        try:
            header, payload, sig_b64 = decode_jwt_parts(token)
        except Exception as e:
            return ValidationResult(False, errors=[f'Malformed JWT: {e}'])
        alg = header.get('alg')
        if alg not in ('HS256', 'RS256'):
            return ValidationResult(False, errors=[f'Rejected alg: {alg}'])
        key = self._keys.get(header.get('kid'))
        if key is None:
            return ValidationResult(False, errors=['Unknown kid'])
        parts = token.split('.')
        expected = b64url_encode(hmac.new(key, f'{parts[0]}.{parts[1]}'.encode(), hashlib.sha256).digest())
        if not hmac.compare_digest(expected, sig_b64):
            errors.append('Signature verification failed')
            return ValidationResult(False, errors=errors)
        if payload.get('iss') != self.issuer: errors.append(f'Issuer mismatch')
        aud = payload.get('aud')
        if (aud if isinstance(aud,str) else '') != self.audience and self.audience not in (aud if isinstance(aud,list) else []):
            errors.append('Audience mismatch')
        if time.time() > payload.get('exp', 0): errors.append('Token expired')
        if nonce and payload.get('nonce') != nonce: errors.append('Nonce mismatch')
        return ValidationResult(len(errors)==0, payload, errors)


validator = OIDCTokenValidator(
    issuer='https://accounts.example.com',
    audience='myapp',
    key_store=KEY_STORE,
)

# Valid
r = validator.validate(token, nonce='nonce_xyz')
print(f'Valid token:    valid={r.valid}  errors={r.errors}')
print(f'  email={r.claims.get("email")}  roles={r.claims.get("roles")}')

# Tampered payload (escalate role)
parts = token.split('.')
h2, p2, _ = decode_jwt_parts(token)
p2['roles'] = ['admin']
tampered = parts[0] + '.' + b64url_encode(json.dumps(p2,separators=(',',':')).encode()) + '.' + parts[2]
r2 = validator.validate(tampered, nonce='nonce_xyz')
print(f'Tampered token: valid={r2.valid}  errors={r2.errors}')

# alg:none attack
h3 = {'alg': 'none', 'typ': 'JWT'}
p3 = {**p2, 'roles': ['admin']}
none_token = b64url_encode(json.dumps(h3).encode()) + '.' + b64url_encode(json.dumps(p3).encode()) + '.'
r3 = validator.validate(none_token)
print(f'alg:none token: valid={r3.valid}  errors={r3.errors}')


## 3. Claim-Based Authorization

In [ ]:
def authorize_from_claims(claims, resource, action):
    roles = set(claims.get('roles', []))
    amr   = set(claims.get('amr',   []))
    policy = {
        'sensitive_data:read':  lambda r,a: 'analyst' in r and 'mfa' in a,
        'sensitive_data:write': lambda r,a: 'admin'   in r and 'mfa' in a,
        'admin_console:read':   lambda r,a: 'admin'   in r and 'mfa' in a,
        'public_api:read':      lambda r,a: True,
    }
    rule = policy.get(f'{resource}:{action}')
    if rule is None: return False, f'No policy for {resource}:{action}'
    if not rule(roles, amr): return False, f'Role/AMR check failed'
    return True, 'Authorized'

tests = [
    ({'roles':['analyst'],'amr':['mfa']}, 'sensitive_data', 'read'),
    ({'roles':['analyst'],'amr':['pwd']}, 'sensitive_data', 'read'),
    ({'roles':['analyst'],'amr':['mfa']}, 'sensitive_data', 'write'),
    ({'roles':['admin'],  'amr':['mfa']}, 'admin_console',  'read'),
]
for claims, res, act in tests:
    ok, reason = authorize_from_claims(claims, res, act)
    print(f'[{"ALLOW" if ok else "DENY "}] {res}:{act}  roles={claims["roles"]} amr={claims["amr"]}  — {reason}')
